In [17]:
# Import required libraries
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from pathlib import Path

base_path = Path.home() / "Downloads" / "ecommerce_preprocessing_stuff_v5" / "data" / "processed"

# Load the preprocessed datasets
train_data = pd.read_csv(base_path/"model_ready_train.csv")
test_data = pd.read_csv(base_path/"model_ready_test.csv")

In [18]:
# Separate features and target variable
X_train = train_data.drop(columns=["churned"])
y_train = train_data["churned"]

X_test = test_data.drop(columns=["churned"])
y_test = test_data["churned"]

In [19]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
RANDOM_STATE = 42

best_rf_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=5,
    n_estimators=200
)

best_gb_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    subsample=1.0,
    n_estimators=300,
    min_samples_leaf=20,
    max_depth=3,
    learning_rate=0.1
)

ensemble_model = VotingClassifier(
    estimators=[
        ("rf", best_rf_model),
        ("gb", best_gb_model)
    ],
    voting="soft"
)

ensemble_model.fit(X_train, y_train)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     min_samples_leaf=2,
                                                     min_samples_split=5,
                                                     n_estimators=200,
                                                     random_state=42)),
                             ('gb',
                              GradientBoostingClassifier(min_samples_leaf=20,
                                                         n_estimators=300,
                                                         random_state=42))],
                 voting='soft')

In [20]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

# Make predictions
y_pred_ensemble = ensemble_model.predict(X_test)

# Get prediction probabilities
y_prob_ensemble = ensemble_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, digits=3))

# Evaluate ensemble
print("Ensemble Accuracy:", f"{accuracy_score(y_test, y_pred_ensemble):.3f}")

print("Ensemble AUC:", f"{roc_auc_score(y_test, y_prob_ensemble):.3f}")


Classification Report:
              precision    recall  f1-score   support

           0      0.922     0.973     0.947      7110
           1      0.924     0.797     0.856      2890

    accuracy                          0.922     10000
   macro avg      0.923     0.885     0.901     10000
weighted avg      0.922     0.922     0.921     10000

Ensemble Accuracy: 0.922
Ensemble AUC: 0.929
